# On-Device & Edge LLMs

**Track:** Production Engineering  
**Toolchain:** llama.cpp · ExecuTorch · CoreML · MLX · WebGPU  
**Objective:** Learn how to deploy Small Language Models (SLMs) natively on phones, laptops, and browsers — entirely offline and with zero API cost.

---

## 🧠 Why On-Device AI?

In 2024, everything ran in the cloud. By 2026, the paradigm is **hybrid AI**: run the easy tasks locally on the user's phone, and only hit the expensive cloud models for complex reasoning.

### The Hybrid Advantage

| Benefit | Cloud LLM (GPT-4) | Edge SLM (Llama-3-8B / Phi-3) |
|---------|------------------|------------------------------|
| **Latency** | 200ms+ (network routing) | <10ms (instant response) |
| **Cost** | ~$0.01 per request | **$0.00** (user pays electricity) |
| **Privacy** | Data leaves the device | **Data never leaves device** |
| **Offline** | Breaks without Wi-Fi | **Works in airplane mode** |
| **Capability** | Deep reasoning, complex coding | Summarization, routing, drafting |

---

## 📑 Table of Contents

* [🧠 Why On-Device AI?](#why-on-device-ai)
  * [The Hybrid Advantage](#the-hybrid-advantage)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Model Format Wars](#1-the-model-format-wars)
  * [Edge Inference Engines](#edge-inference-engines)
* [2 · The Hybrid Architecture (Local + Cloud)](#2-the-hybrid-architecture-local-cloud)
  * [Example User Flow:](#example-user-flow)
* [Knowledge Check](#knowledge-check)
  * [Q1: Model Format](#q1-model-format)
  * [Q2: Hybrid Router](#q2-hybrid-router)
  * [Q3: Battery](#q3-battery)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Design a hybrid routing strategy for a note-taking app. Which tasks go local vs cloud?](#exercise-1-design-a-hybrid-routing-strategy-for-a-note-taking-app-which-tasks-go-local-vs-cloud)
  * [Exercise 2: Calculate if a Phi-3 Mini (3.8B) at Q4 fits on a phone with 6GB RAM (OS uses 2GB).](#exercise-2-calculate-if-a-phi-3-mini-38b-at-q4-fits-on-a-phone-with-6gb-ram-os-uses-2gb)
* [3 · WebGPU: LLMs in the Browser](#3-webgpu-llms-in-the-browser)
  * [Why WebGPU Changes Everything:](#why-webgpu-changes-everything)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Cloud AI has a latency floor and privacy risks. Seniors design hybrid architectures where smaller, quantized models (SLMs) run instantly and cost-effectively directly on the user's phone, laptop, or browser.

**Mental Model:** Edge AI is like keeping a pocket calculator for simple math (instant, localized) instead of calling a remote supercomputer. Models must be squeezed (quantized) to fit into 3GB of phone RAM.

**Common Junior Pitfall:** Trying to run full PyTorch tensors on an iPhone. Mobile chips (NPU/Neural Engine) require specific highly-optimized edge formats like GGUF, CoreML, or ExecuTorch.

---


In [ ]:
# Cell 1 — Install
!pip install -q huggingface_hub

## 1 · The Model Format Wars

If you try to load a PyTorch `.bin` or `.safetensors` file on an iPhone, the phone will crash. Mobile processors (like the Apple Neural Engine or Snapdragon NPU) need highly optimized, quantized formats.

### Edge Inference Engines

| Format / Engine | Target Hardware | Use Case |
|-----------------|-----------------|----------|
| **GGUF** (`llama.cpp`) | CPU / Apple Silicon (Mac) | Standard for running local models on laptops. Great CPU performance. |
| **CoreML** | iOS / Apple Neural Engine | Natively optimized for iPhones and iPads. Lowest battery drain. |
| **ExecuTorch** (PyTorch) | Android / iOS / Wearables | Meta's framework for deploying Llama to mobile devices. |
| **MLX** | Apple Silicon (Mac) | Apple's native framework for high-speed ML on M-series chips. |
| **ONNX Runtime** | Windows Copilot+ PCs | Standard for Windows NPUs. |
| **WebGPU** | Browser (Chrome/Safari) | Running models entirely in the user's web browser (`transformers.js`). |

In [ ]:
# Cell 2 — Calculating mobile VRAM constraints

def estimate_mobile_vram(params_billions, precision_bits):
    """Calculate how much RAM/VRAM a model needs on a phone."""
    bytes_per_param = precision_bits / 8
    model_size_gb = (params_billions * 1e9 * bytes_per_param) / 1024**3
    
    # Add ~20% overhead for KV Cache and runtime
    required_ram_gb = model_size_gb * 1.2
    return round(required_ram_gb, 2)

phones = [
    {'name': 'iPhone 15 Pro', 'ram': 8},
    {'name': 'Pixel 8 Pro', 'ram': 12},
    {'name': 'Budget Android', 'ram': 4}
]

models = [
    {'name': 'Llama-3-70B', 'params': 70, 'bits': 16, 'type': 'Cloud Only'},
    {'name': 'Llama-3-8B', 'params': 8, 'bits': 4, 'type': 'Edge/Laptop'},
    {'name': 'Phi-3-Mini', 'params': 3.8, 'bits': 4, 'type': 'Mobile'},
    {'name': 'Gemma-2B', 'params': 2, 'bits': 4, 'type': 'Wearable/IoT'}
]

print('📱 Edge Model VRAM Calculator')
print('=' * 60)

for model in models:
    req_ram = estimate_mobile_vram(model['params'], model['bits'])
    print(f'\n{model["name"]} ({model["type"]})')
    print(f'  - Precision: {model["bits"]}-bit')
    print(f'  - Required VRAM/RAM: ~{req_ram} GB')
    
    compatible = [p['name'] for p in phones if p['ram'] > req_ram + 1] # Need 1GB for OS
    print(f'  - Runs on: {", ".join(compatible) if compatible else "None of these"}')

print('\n💡 The "Mobile AI" sweet spot requires models under 4B parameters quantized')
print('   to 4-bit, keeping memory usage well under 3GB.')

In [ ]:
# Cell -- GGUF model selection helper
# Helps choose the right quantization for your hardware

def recommend_gguf(available_ram_gb, use_case='general'):
    """Recommend the best GGUF quantization for available RAM."""
    options = [
        ('Q2_K', 2, 'Smallest, noticeable quality loss'),
        ('Q4_K_M', 4, 'Best balance of quality and size'),
        ('Q5_K_M', 5, 'Near-original quality'),
        ('Q8_0', 8, 'Minimal quality loss'),
        ('F16', 16, 'Original quality, largest'),
    ]
    print(f'Available RAM: {available_ram_gb}GB')
    print(f'{"Quant":<10} {"Bits":<6} {"7B Size":<10} {"Fits?":<6} {"Notes"}')
    print('-' * 55)
    best = None
    for name, bits, notes in options:
        size = 7 * bits / 8 * 1.2  # 7B model + 20% overhead
        fits = size < available_ram_gb - 1  # Leave 1GB for OS
        icon = 'YES' if fits else 'NO'
        print(f'{name:<10} {bits:<6} {size:.1f}GB     {icon:<6} {notes}')
        if fits: best = name
    print(f'\nRecommended: {best}')

print('Phone (6GB available):')
recommend_gguf(6)
print('\nLaptop (16GB available):')
recommend_gguf(16)


---

## 2 · The Hybrid Architecture (Local + Cloud)

The best applications don't force everything onto the phone. They use a **router/orchestrator** on the device to decide whether to handle a request locally or send it to the cloud.

### Example User Flow:
1. **User:** "Summarize this 3-paragraph email." → **Local Phi-3 router** handles it instantly. Zero cost.
2. **User:** "Write a Python script to parse a 10MB JSON and upload it to S3." → **Local router** determines task complexity is too high → Forwards request to **Cloud GPT-4o**.

In [ ]:
# Cell 3 — Simulating an On-Device Hybrid Router
import time

class HybridAIRouter:
    """Runs on the edge device to route requests efficiently."""
    
    def __init__(self):
        self.local_model_name = "Phi-3-Mini (ON-DEVICE)"
        self.cloud_model_name = "GPT-4o (CLOUD)"
        self.battery_level = 100
    
    def determine_complexity(self, prompt):
        """Very fast heuristic to decide routing."""
        # In reality, this might be a tiny classifying neural net or regex rules
        code_keywords = ['python', 'react', 'sql', 'algorithm']
        reasoning_keywords = ['analyze', 'calculate', 'compare', 'logic']
        
        prompt_lower = prompt.lower()
        is_code = any(kw in prompt_lower for kw in code_keywords)
        is_reasoning = any(kw in prompt_lower for kw in reasoning_keywords)
        is_long = len(prompt) > 2000
        
        return is_code or is_reasoning or is_long
        
    def process(self, prompt, is_airplane_mode=False):
        print(f'\n👤 User: "{prompt}"')
        
        if is_airplane_mode:
            print(f'  ✈️ Airplane Mode Active — Forcing Local Execution')
            return self._run_local(prompt)
            
        if self.battery_level < 15:
            print(f'  🪫 Low Battery Alert — Offloading to Cloud to save power')
            return self._run_cloud(prompt)
            
        needs_cloud = self.determine_complexity(prompt)
        
        if needs_cloud:
            return self._run_cloud(prompt)
        else:
            return self._run_local(prompt)
            
    def _run_local(self, prompt):
        print(f'  🧠 Routing to {self.local_model_name}')
        print(f'  ⏱️ Latency: ~10ms | Cost: $0.00 | Privacy: Complete')
        self.battery_level -= 1 # Local ML drains battery
        return "[Local response generated]"
        
    def _run_cloud(self, prompt):
        print(f'  ☁️ Routing to {self.cloud_model_name}')
        print(f'  ⏱️ Latency: ~800ms | Cost: $0.02 | Privacy: Sent to server')
        return "[Cloud response generated]"


router = HybridAIRouter()
print('🔀 Edge Hybrid Router Dashboard')
print('=' * 60)

prompts = [
    "Summarize this article intro: The new iPhone features a titanium body...",
    "Draft a polite decline email to John for the meeting tomorrow.",
    "Write a Python script to perform sentiment analysis on 1M tweets."
]

for prompt in prompts:
    router.process(prompt)

# Test Airplane mode
router.process("Analyze the architectural flaws in microservices...", is_airplane_mode=True)


print('\n💡 The Hybrid approach gives you the instant latency and privacy')
print('   of local models for 80% of tasks, while leaning on the cloud')
print('   for heavy lifting.')

---
## Knowledge Check

### Q1: Model Format
Why can't you load a PyTorch `.safetensors` file on an iPhone?

<details><summary>Answer</summary>

iPhones use the Apple Neural Engine (ANE), which requires CoreML format. PyTorch tensors are designed for NVIDIA GPUs with CUDA. The ANE doesn't understand PyTorch operations. You must convert: PyTorch -> CoreML (using `coremltools`) or use ExecuTorch.
</details>

### Q2: Hybrid Router
Your hybrid router sends 'Write a Python web scraper' to the cloud. Why not handle it locally?

<details><summary>Answer</summary>

Code generation requires strong reasoning and long-context understanding. Small edge models (3-4B params) produce unreliable code with bugs. Cloud models (70B+) have much better code quality. The router correctly identifies this as a complex task.
</details>

### Q3: Battery
Running a 3B model locally uses significant battery. How would you optimize for battery life?

<details><summary>Answer</summary>

Use more aggressive quantization (Q2 vs Q4), limit max tokens, cache frequent responses, batch processing when plugged in, and route to cloud when battery < 20%. Also use the Neural Engine (ANE) instead of CPU -- it's 10x more power efficient.
</details>


---
## Practical Practice

### Exercise 1: Design a hybrid routing strategy for a note-taking app. Which tasks go local vs cloud?
### Exercise 2: Calculate if a Phi-3 Mini (3.8B) at Q4 fits on a phone with 6GB RAM (OS uses 2GB).


In [ ]:
# ===== SOLUTIONS =====
print('Ex 1: Note-Taking Hybrid Router')
tasks = [
    ('Autocomplete sentence', 'LOCAL', 'Simple continuation, low latency needed'),
    ('Summarize 1-page note', 'LOCAL', 'Small context, Phi-3 handles well'),
    ('Generate meeting notes from audio', 'CLOUD', 'Requires ASR + long context'),
    ('Fix grammar/spelling', 'LOCAL', 'Pattern matching, small model sufficient'),
    ('Research and draft a report', 'CLOUD', 'Needs web search + complex reasoning'),
]
for task, where, reason in tasks:
    print(f'  {where:5} | {task:40} | {reason}')

print('\nEx 2: Phi-3 Mini on 6GB phone')
params_b = 3.8
bits = 4
model_gb = params_b * bits / 8
overhead = model_gb * 0.2
total = model_gb + overhead
available = 6 - 2  # 6GB RAM - 2GB OS
print(f'  Model: {model_gb:.1f}GB + overhead: {overhead:.1f}GB = {total:.1f}GB')
print(f'  Available: {available}GB')
print(f'  Fits: {"YES" if total < available else "NO"} ({available - total:.1f}GB headroom)')


---

## 3 · WebGPU: LLMs in the Browser

Perhaps the biggest shift in 2026 is **In-Browser inference**. Thanks to the WebGPU standard, JavaScript can access the user's local GPU directly.

Using libraries like `transformers.js` or `WebLLM`, you can load a 4-bit Llama-3-8B model directly inside Chrome or Safari.

### Why WebGPU Changes Everything:
1. **Zero Install:** Users don't need Python or Docker. Just visit a URL.
2. **Zero Hosting Costs:** You pay for static file hosting (S3/Vercel) to serve the model weights (e.g., a 4GB file). The inference cost ($0) is pushed entirely to the user's laptop.
3. **Ultimate Privacy:** The user chats with the AI, and no data ever leaves their browser tab.

---

## 🎯 Summary & Key Takeaways

- **SLMs rule the edge:** Models under 8B parameters, quantized to 4-bit (GGUF/CoreML), are the standard for mobile and laptop deployment.
- **Hybrid Routing:** The ultimate AI architecture is an on-device orchestrator that defaults to local inference for privacy/speed, but falls back to cloud APIs for complex reasoning.
- **WebGPU:** Allows full offline LLM inferencing directly inside a web browser, eliminating massive API bills for developers.

**End of Module.**

In [ ]:
# Cell -- WebGPU browser inference example (HTML/JS)
# This generates a working HTML file that runs an LLM in the browser

webgpu_html = '''
<!DOCTYPE html>
<html>
<head><title>Browser LLM (WebGPU)</title></head>
<body>
  <h2>In-Browser LLM with WebGPU</h2>
  <textarea id="prompt" rows="3" cols="60">Explain quantum computing in one sentence.</textarea><br>
  <button onclick="generate()">Generate (runs locally!)</button>
  <pre id="output">Output will appear here...</pre>
  <script type="module">
    // In production, use: import { pipeline } from "@xenova/transformers";
    // The model downloads once (~4GB) then runs entirely in your browser.
    // Zero API costs. Zero data leaves the device.
    window.generate = async function() {
      document.getElementById("output").textContent = "Loading model...";
      // const pipe = await pipeline("text-generation", "Xenova/phi-2", {device: "webgpu"});
      // const result = await pipe(document.getElementById("prompt").value);
      document.getElementById("output").textContent = "[Model would generate here in a real browser]";
    }
  </script>
</body>
</html>
'''
with open('browser_llm.html', 'w') as f:
    f.write(webgpu_html)
print('WebGPU browser LLM demo written to browser_llm.html')
print('Key technologies:')
print('  - transformers.js: HuggingFace models in JavaScript')
print('  - WebGPU: Direct GPU access from the browser')
print('  - ONNX/WASM: Cross-platform model format')
print('\nCost comparison:')
print('  Cloud API: $0.01/request x 1M requests = $10,000/month')
print('  WebGPU:    $0/request (static hosting only ~$5/month)')
